# 02 · Checks on the half-hourly panel

Verification behind `negprice/processing.py`. The panel itself is built by `make data` (`data/processed/panel.parquet`); this notebook only checks it. Each section says what is tested, why, and the result.

Column definitions: `references/data_dictionary.md`.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

import negprice.processing as proc
from negprice.config import INTERIM_DATA_DIR, RAW_DATA_DIR, REGIONS


def read(name: str) -> pd.DataFrame:
    return pd.read_parquet(INTERIM_DATA_DIR / f"{name}.parquet")


def wape(actual: pd.Series, forecast: pd.Series) -> float:
    """Weighted absolute percentage error: sum|actual - forecast| / sum|actual|."""
    return (actual - forecast).abs().sum() / actual.abs().sum()

## 1. Timestamp conventions

**Why:** six sources are joined on time. One source labelled by interval start, or carrying daylight saving, would shift every join by 30–60 minutes without raising any error.

**Expected:** naive timestamps, NEM time (UTC+10 all year), labelling the **end** of each interval.

### A. Every datetime column

In [2]:
for f in sorted(INTERIM_DATA_DIR.glob("*.parquet")):
    df = pd.read_parquet(f)
    print(f"\n== {f.name}  ({len(df):,} rows)")
    for c in df.columns:
        s = df[c]
        if pd.api.types.is_datetime64_any_dtype(s):
            mins = sorted(s.dt.minute.dropna().unique())
            print(f"  {c:20} {str(s.dtype):22} {s.min()} → {s.max()}  minutes={mins[:12]}")


== dispatchprice.parquet  (1,577,664 rows)
  SETTLEMENTDATE       datetime64[us]         2021-10-01 00:05:00 → 2026-10-01 00:00:00  minutes=[np.int32(0), np.int32(5), np.int32(10), np.int32(15), np.int32(20), np.int32(25), np.int32(30), np.int32(35), np.int32(40), np.int32(45), np.int32(50), np.int32(55)]

== dispatchregionsum.parquet  (1,577,664 rows)
  SETTLEMENTDATE       datetime64[us]         2021-10-01 00:05:00 → 2026-10-01 00:00:00  minutes=[np.int32(0), np.int32(5), np.int32(10), np.int32(15), np.int32(20), np.int32(25), np.int32(30), np.int32(35), np.int32(40), np.int32(45), np.int32(50), np.int32(55)]

== predispatch_price.parquet  (258,480 rows)
  RUN_DATETIME         datetime64[us]         2021-09-30 13:00:00 → 2026-09-29 13:00:00  minutes=[np.int32(0)]
  DATETIME             datetime64[us]         2021-10-01 00:30:00 → 2026-10-01 00:00:00  minutes=[np.int32(0), np.int32(30)]

== predispatch_regionsum.parquet  (258,480 rows)
  RUN_DATETIME         datetime64[us]         20

**Result:** all columns naive `datetime64[us]`. Dispatch runs 00:05 → 00:00 in 5-minute steps (interval end). Half-hourly interval columns start at 00:30 with minutes {0, 30}. Predispatch runs at 13:00 only; rooftop PV forecast versions are ≤ 12:30. Row counts reconcile exactly with 3 regions × 1,826 days (predispatch minus the Oct 2022 gap).

### B. Physical test: does solar noon sit where it should, all year?

The sun ignores clock conventions, so the centre of the rooftop PV curve is a solar-noon detector. With interval-end labels it should read solar noon + 15 min; daylight saving would show as a +1 h jump in Oct–Mar.

In [3]:
pv_noon = read("rooftop_pv_actual")
pv_noon = pv_noon[pv_noon.TYPE == "MEASUREMENT"].dropna(subset=["POWER"])
t = pv_noon.INTERVAL_DATETIME
pv_noon["tod"] = t.dt.hour + t.dt.minute / 60
pv_noon["month"] = t.dt.month

# Power-weighted mean time of day = centre of the solar curve
pv_noon["w"] = pv_noon.tod * pv_noon.POWER
g = pv_noon.groupby(["REGIONID", "month"])[["w", "POWER"]].sum()
centre = (g.w / g.POWER).unstack("REGIONID")
print(centre.round(2))

REGIONID   NSW1    SA1   VIC1
month                        
1         12.29  13.22  12.77
2         12.36  13.30  12.84
3         12.32  13.22  12.75
4         12.16  13.05  12.53
5         12.11  13.00  12.50
6         12.16  13.03  12.57
7         12.27  13.15  12.63
8         12.22  13.09  12.62
9         12.10  12.98  12.45
10        11.89  12.86  12.34
11        11.89  12.83  12.35
12        12.09  13.01  12.54


**Result: passed.** Annual means NSW 12.16 h, VIC 12.57 h, SA 13.08 h: solar noon (from longitude) + ~15 min, and SA − NSW ≈ 55 min, VIC − NSW ≈ 25 min. No jump between Oct–Mar and Apr–Sep (no daylight saving). The ±15 min month-to-month wobble, highest in February and lowest in Oct–Nov, is the equation of time.

## 2. Dispatch price: regions, intervention, duplicates

**Why:** when AEMO intervenes in the market, dispatch runs twice. Only the pricing run (`INTERVENTION = 0`) sets the price participants are paid; keeping both rows would double-count those intervals.

In [4]:
dp = read("dispatchprice")
print(dp.columns.tolist())
print(dp.REGIONID.value_counts())
print("duplicate keys:", dp.duplicated(["SETTLEMENTDATE", "REGIONID"]).sum())
print("missing RRP:", dp.RRP.isna().sum())
print(dp.groupby("REGIONID").RRP.describe().round(1))

['SETTLEMENTDATE', 'REGIONID', 'RRP']
REGIONID
NSW1    525888
SA1     525888
VIC1    525888
Name: count, dtype: int64
duplicate keys: 0
missing RRP: 0
             count   mean    std     min   25%   50%    75%      max
REGIONID                                                            
NSW1      525888.0  116.9  429.4 -1000.0  57.0  84.0  125.5  20300.0
SA1       525888.0   98.7  480.0 -1000.0   0.0  67.6  136.0  20300.0
VIC1      525888.0   78.9  262.4 -1000.0   7.5  60.6  118.0  19069.7


**Result:** 525,888 rows per region (1,826 days × 288), no duplicates, no missing values. Minimum −$1,000 (market floor); maximum $20,300 (market price cap, FY2026).

`INTERVENTION` was dropped by `dataset.py`, so the check below goes back to the NEMOSIS cache. **The cache stores every column as text**, so it is converted first.

In [5]:
files = sorted((RAW_DATA_DIR / "nemosis_cache").glob("*DISPATCHPRICE*.parquet"))
raw = pd.concat((pd.read_parquet(f) for f in files), ignore_index=True)
raw = raw[raw.REGIONID.isin(REGIONS)].assign(
    SETTLEMENTDATE=lambda d: pd.to_datetime(d.SETTLEMENTDATE, format="%Y/%m/%d %H:%M:%S").astype("datetime64[us]"),
    INTERVENTION=lambda d: pd.to_numeric(d.INTERVENTION),
    RRP=lambda d: pd.to_numeric(d.RRP),
)
print(raw.groupby(["REGIONID", "INTERVENTION"]).size())
print(raw.PRICE_STATUS.value_counts(dropna=False))

# Where an intervention run exists: does our file hold the pricing-run (INTERVENTION = 0) price?
iv = raw[raw.INTERVENTION == 1][["SETTLEMENTDATE", "REGIONID"]]
check = (iv.merge(raw[raw.INTERVENTION == 0], on=["SETTLEMENTDATE", "REGIONID"])
           .merge(dp, on=["SETTLEMENTDATE", "REGIONID"], suffixes=("_iv0", "_ours")))
print("intervention intervals in the study window:", len(check))
print("ours == pricing-run price:", (check.RRP_iv0 == check.RRP_ours).mean())

# Share of 5-min prices exactly $0
print((dp.RRP == 0).groupby(dp.REGIONID).mean().round(4))

REGIONID  INTERVENTION
NSW1      0               534528
          1                 3487
SA1       0               534528
          1                 3487
VIC1      0               534528
          1                 3487
dtype: int64
PRICE_STATUS
FIRM    1614045
Name: count, dtype: int64
intervention intervals in the study window: 10461
ours == pricing-run price: 1.0
REGIONID
NSW1    0.0050
SA1     0.0069
VIC1    0.0059
Name: RRP, dtype: float64


**Result:** 3,487 intervention intervals per region; all prices `FIRM`. `dataset.py` keeps `INTERVENTION == 0` before dropping the column, so the match should be 1.0. The cache also holds Sep 2021 (NEMOSIS downloads whole months), which is outside the study window.

0.5–0.7% of 5-min prices are exactly $0, so the target "< 0" vs "≤ 0" changes base rates by under 1 point.

## 3. Resample to half-hours

**Rule:** the half-hour ending HH:30 holds the six 5-min intervals ending HH:05 … HH:30 (`ceil("30min")`).

### Price

In [6]:
hh = proc.price_to_half_hourly(dp)
print(len(hh), "expected", 3 * 87_648)
print(hh.n_5min.value_counts())
print(hh.HH_END.min(), "→", hh.HH_END.max(), hh.HH_END.dtype)
print(hh.groupby("REGIONID").agg(
    mean_neg=("rrp_mean", lambda s: (s < 0).mean()),
    any_neg=("rrp_min", lambda s: (s < 0).mean()),
).round(3))

262944 expected 262944
n_5min
6    262944
Name: count, dtype: int64
2021-10-01 00:30:00 → 2026-10-01 00:00:00 datetime64[us]
          mean_neg  any_neg
REGIONID                   
NSW1         0.067    0.109
SA1          0.252    0.299
VIC1         0.213    0.261


**Result:** 262,944 half-hours, all with exactly 6 intervals. Base rates match notebook 01, computed by an independent code path:

| Region | Half-hour mean < $0 | Any 5-min interval < $0 |
|---|---|---|
| SA | 25.2% | 29.9% |
| VIC | 21.3% | 26.1% |
| NSW | 6.7% | 10.9% |

### Demand and supply (`DISPATCHREGIONSUM`)

In [7]:
rs = read("dispatchregionsum")
print(rs.columns.tolist())
print(rs.isna().sum())

hr = proc.regionsum_to_half_hourly(rs)
print(len(hr), "| same keys as price:",
      hr[["REGIONID", "HH_END"]].equals(hh[["REGIONID", "HH_END"]]))
print(hr.groupby("REGIONID").TOTALDEMAND.describe().round(0))

['SETTLEMENTDATE', 'REGIONID', 'TOTALDEMAND', 'AVAILABLEGENERATION', 'NETINTERCHANGE', 'TOTALINTERMITTENTGENERATION', 'DEMAND_AND_NONSCHEDGEN', 'UIGF']
SETTLEMENTDATE                 0
REGIONID                       0
TOTALDEMAND                    0
AVAILABLEGENERATION            0
NETINTERCHANGE                 0
TOTALINTERMITTENTGENERATION    0
DEMAND_AND_NONSCHEDGEN         0
UIGF                           0
dtype: int64
262944 | same keys as price: True
            count    mean     std     min     25%     50%     75%      max
REGIONID                                                                  
NSW1      87648.0  7509.0  1400.0  2582.0  6565.0  7389.0  8304.0  13716.0
SA1       87648.0  1295.0   437.0  -284.0  1106.0  1354.0  1543.0   3309.0
VIC1      87648.0  4820.0  1090.0  1244.0  4102.0  4698.0  5451.0  10640.0


In [8]:
# Lowest demand per region: real events or glitches?
for r in REGIONS:
    print(hr[hr.REGIONID == r].nsmallest(5, "TOTALDEMAND")[["HH_END", "TOTALDEMAND"]])

                    HH_END  TOTALDEMAND
161883 2025-12-25 14:00:00  -283.663333
161882 2025-12-25 13:30:00  -276.575000
161884 2025-12-25 14:30:00  -256.575000
161881 2025-12-25 13:00:00  -242.573333
141145 2024-10-19 13:00:00  -218.828333
                    HH_END  TOTALDEMAND
249625 2025-12-27 13:00:00  1243.511667
249626 2025-12-27 13:30:00  1278.446667
249624 2025-12-27 12:30:00  1290.001667
249627 2025-12-27 14:00:00  1338.751667
249623 2025-12-27 12:00:00  1339.646667
                   HH_END  TOTALDEMAND
59256 2025-02-16 12:30:00  2581.838333
59257 2025-02-16 13:00:00  2595.091667
59258 2025-02-16 13:30:00  2620.573333
59255 2025-02-16 12:00:00  2633.410000
59259 2025-02-16 14:00:00  2697.060000


**Result:** no missing values; same keys as price. The lowest demand values are smooth midday blocks on sunny, low-activity days: NSW Sunday 16 Feb 2025 (2,582 MW), VIC Saturday 27 Dec 2025, SA Christmas Day 2025 (−284 MW). Real, kept.

## 4. Rooftop PV actuals

**Leakage question:** if values were revised long after the interval, the archive holds the revised number, not the one known at the time.

In [9]:
pva = proc.pv_actual_half_hourly(read("rooftop_pv_actual"))
print(len(pva), "rows | expected", 3 * 87_648 - 6)
print("empty values:", pva.pv_actual_mw.isna().sum())

lag = (pva.pv_actual_known_at - pva.HH_END).dt.total_seconds() / 3600
print(lag.describe().round(2))
print("share published > 1 h after interval:", (lag > 1).mean().round(4))
print("share > 24 h:", (lag > 24).mean().round(4))

262938 rows | expected 262938
empty values: 28
count    262938.00
mean          0.32
std           0.00
min           0.31
25%           0.32
50%           0.32
75%           0.32
max           0.60
dtype: float64
share published > 1 h after interval: 0.0
share > 24 h: 0.0


**Result:** every value published 19–36 min after its interval, none later, so stored values are the values known at the time; PV actuals up to 13:00 on D-1 are a safe feature. 34 half-hours have no measurement (6 absent rows + empty values); they stay missing.

## 5. Master grid

Every (region, half-hour) that should exist. Each source is left-joined onto it, so gaps show up as missing values instead of silently missing rows.

In [10]:
grid = proc.half_hour_grid()
print(len(grid), "expected 262944")
print(grid.HH_END.min(), "→", grid.HH_END.max(), grid.HH_END.dtype)
print(grid.groupby(["REGIONID", "DAY"]).size().value_counts())
print("days:", grid.DAY.nunique(), "| first/last:", grid.DAY.min().date(), grid.DAY.max().date())
print("keys match price:",
      grid.sort_values(["REGIONID", "HH_END"])[["REGIONID", "HH_END"]]
          .reset_index(drop=True).equals(hh[["REGIONID", "HH_END"]]))

262944 expected 262944
2021-10-01 00:30:00 → 2026-10-01 00:00:00 datetime64[us]
48    5478
Name: count, dtype: int64
days: 1826 | first/last: 2021-10-01 2026-09-30
keys match price: True


**Result:** 262,944 rows; all 5,478 region-days (3 × 1,826) have exactly 48 half-hours.

## 6. Join all sources + leakage check

In [11]:
panel = proc.build_panel(
    grid, hh, hr, pva,
    read("predispatch_price"), read("predispatch_regionsum"),
    read("rooftop_pv_forecast"), read("silo_weather"),
)
print(panel.shape)
print(panel.columns.tolist())
na = panel.isna().sum()
print(na[na > 0])

# Leakage: every forecast must be known by issue time (13:00 on D-1 = DAY - 11 h)
issue = panel.DAY - pd.Timedelta(hours=11)
for col in ["pd_known_at", "pvf_known_at"]:
    ok = panel[col].isna() | (panel[col] <= issue)
    print(col, "all ≤ issue time:", ok.all())

(262944, 35)
['REGIONID', 'HH_END', 'DAY', 'rrp_mean', 'rrp_min', 'n_neg_5min', 'n_5min', 'totaldemand', 'availablegeneration', 'netinterchange', 'totalintermittentgeneration', 'demand_and_nonschedgen', 'uigf', 'pv_actual_mw', 'pv_actual_known_at', 'pd_known_at', 'pd_rrp', 'pd_totaldemand', 'pd_availablegeneration', 'pd_netinterchange', 'pd_demand_and_nonschedgen', 'pd_uigf', 'pd_ss_solar_uigf', 'pd_ss_wind_uigf', 'pvf_known_at', 'pvf_powermean', 'pvf_powerpoe50', 'pvf_powerpoelow', 'pvf_powerpoehigh', 'silo_daily_rain', 'silo_max_temp', 'silo_min_temp', 'silo_radiation', 'silo_vp', 'silo_known_at']
pv_actual_mw                   34
pv_actual_known_at              6
pd_known_at                  4464
pd_rrp                       4464
pd_totaldemand               4464
pd_availablegeneration       4464
pd_netinterchange            4464
pd_demand_and_nonschedgen    4464
pd_uigf                      4464
pd_ss_solar_uigf             4464
pd_ss_wind_uigf              4464
dtype: int64
pd_kno

**Result:** 262,944 rows, no row multiplication (`validate=` on every merge). Missing values are exactly the known ones: 4,464 = 3 × 1,488 predispatch (Oct 2022 gap) and 34 PV actuals. Every forecast was known by 13:00 on D-1.

## 7. Quality checks: values wrong but present

Structure (duplicates, gaps, timezone) is already checked. These look for frozen sensors, broken forecast periods and physically impossible values.

In [12]:
p = panel
hour = p.HH_END.dt.hour

# 1. Completeness of every half-hour
print("n_5min all 6:", (p.n_5min == 6).all())

# 2. Price floor, and max per financial year (the cap rises each July)
print("floor respected:", (p.rrp_min >= -1000).all())
fy = (p.HH_END - pd.Timedelta("30min")).dt.to_period("Q-JUN").dt.qyear
print(p.groupby(fy).rrp_mean.max().round(0))

# 3. PV physics: no negatives, 0 at night, forecast quantiles ordered
night = hour.isin([22, 23, 0, 1, 2, 3])
print("PV actual min:", p.pv_actual_mw.min(), "| max at night:", p.loc[night, "pv_actual_mw"].max())
ordered = (p.pvf_powerpoelow <= p.pvf_powerpoe50) & (p.pvf_powerpoe50 <= p.pvf_powerpoehigh)
print("PV forecast quantiles ordered:", ordered.mean().round(5))


# 4. Frozen values: longest run of identical consecutive values (half-hours)
def longest_run(s):
    return s.groupby((s != s.shift()).cumsum()).size().max()


for col in ["totaldemand", "pd_rrp", "pd_totaldemand", "pv_actual_mw"]:
    s = p[col] if col != "pv_actual_mw" else p[col].where(hour.between(10, 15))   # PV: daytime only
    print(col, p.assign(s=s).dropna(subset=["s"]).groupby("REGIONID").s.apply(longest_run).to_dict())

# 5. Forecast vs actual by year: a jump flags a broken period
yr = p.DAY.dt.year
print(p.groupby([yr, "REGIONID"]).apply(
    lambda d: pd.Series({
        "demand_WAPE": wape(d.totaldemand, d.pd_totaldemand),
        "pv_WAPE": wape(d.pv_actual_mw, d.pvf_powerpoe50),
    })).unstack("REGIONID").round(3))

n_5min all 6: True
floor respected: True
HH_END
2022    15100.0
2023    13351.0
2024    16600.0
2025    17500.0
2026    20300.0
2027     5757.0
Name: rrp_mean, dtype: float64
PV actual min: 0.0 | max at night: 0.0
PV forecast quantiles ordered: 1.0
totaldemand {'NSW1': 2, 'SA1': 2, 'VIC1': 1}
pd_rrp {'NSW1': 137, 'SA1': 102, 'VIC1': 57}
pd_totaldemand {'NSW1': 2, 'SA1': 2, 'VIC1': 1}
pv_actual_mw {'NSW1': 1, 'SA1': 1, 'VIC1': 1}
         demand_WAPE               pv_WAPE              
REGIONID        NSW1    SA1   VIC1    NSW1    SA1   VIC1
DAY                                                     
2021           0.028  0.080  0.036   0.108  0.119  0.105
2022           0.023  0.053  0.029   0.102  0.137  0.119
2023           0.024  0.056  0.034   0.086  0.119  0.121
2024           0.028  0.058  0.037   0.098  0.109  0.112
2025           0.029  0.060  0.040   0.100  0.106  0.117
2026           0.026  0.058  0.038   0.101  0.123  0.131


**Result:**

- Completeness, floor, PV physics and quantile order: all pass.
- Yearly price maxima equal each financial year's market price cap ($15,100 FY22 → $20,300 FY26).
- Forecast error stable across years (demand 2–8%, PV 9–14%): no broken year.
- **One finding:** `pd_rrp` frozen for up to 137 consecutive half-hours (NSW). Investigated below.

## 8. Problem periods

### A. Where are the frozen prices?

In [13]:
def runs(df, col, min_len=12):
    """Runs of identical consecutive values, at least min_len half-hours long."""
    out = []
    for r, d in df.groupby("REGIONID"):
        g = (d[col] != d[col].shift()).cumsum()
        rl = d.groupby(g).agg(start=("HH_END", "first"), end=("HH_END", "last"),
                              value=(col, "first"), n=(col, "size"))
        out.append(rl[rl.n >= min_len].assign(REGIONID=r))
    return pd.concat(out).sort_values("n", ascending=False).reset_index(drop=True)


print(runs(p, "pd_rrp").head(15).to_string())
print(runs(p, "rrp_mean").head(15).to_string())

                 start                 end     value    n REGIONID
0  2022-06-15 14:30:00 2022-06-18 10:30:00  15100.00  137     NSW1
1  2022-06-16 06:30:00 2022-06-18 09:00:00  15100.00  102      SA1
2  2022-06-17 06:30:00 2022-06-18 10:30:00  15100.00   57     VIC1
3  2022-06-22 02:30:00 2022-06-23 00:00:00  15100.00   44      SA1
4  2022-06-16 07:30:00 2022-06-17 04:00:00  15100.00   42     VIC1
5  2022-06-14 16:00:00 2022-06-15 11:30:00  15100.00   40     NSW1
6  2022-06-22 07:00:00 2022-06-23 00:00:00  15100.00   35     VIC1
7  2022-06-21 15:30:00 2022-06-22 04:00:00  15100.00   26     NSW1
8  2022-06-20 16:30:00 2022-06-21 04:00:00  15100.00   24      SA1
9  2022-06-18 15:30:00 2022-06-19 02:30:00  15100.00   23     NSW1
10 2022-06-20 16:00:00 2022-06-21 02:30:00  15100.00   22     NSW1
11 2022-04-06 06:30:00 2022-04-06 17:00:00    299.99   22     NSW1
12 2022-06-21 15:00:00 2022-06-22 01:00:00  15100.00   21      SA1
13 2022-01-11 11:30:00 2022-01-11 21:30:00  15100.00   21     

**Result:** the long runs are June 2022: AEMO's forecast frozen at the $15,100 cap while actual prices sat at the **$300 administered price cap** (energy crisis, then market suspension). Shorter runs elsewhere (SA 11 Jan 2022 cap forecast, NSW 6 Apr 2022 at $299.99, VIC 15–16 Jan 2026 at −$2.70 overnight) are genuine market outcomes and are kept.

### B. Administered-price window (`admin_price` flag)

Flag set in `config.ADMIN_PRICE_DAYS`. Kept for training (demand, PV and weather were real); excluded from backtest scoring and the battery dollar figure.

In [14]:
p = proc.add_admin_price_flag(panel)
print("flagged half-hours:", p.admin_price.sum(), f"({p.admin_price.mean():.2%})")

june = p[p.DAY.between("2022-06-01", "2022-06-30")]
for col, val in [("rrp_mean", 300), ("pd_rrp", 15100)]:
    print(f"\n{col} = {val}")
    print(june[june[col] == val].groupby("REGIONID").HH_END.agg(["min", "max"]).to_string())

edge = p[p.DAY.between("2022-06-08", "2022-06-28")]
print("\n", edge.groupby(["DAY", "REGIONID"]).rrp_mean.mean().unstack().round(0).to_string())

flagged half-hours: 1728 (0.66%)

rrp_mean = 300
                         min                 max
REGIONID                                        
NSW1     2022-06-13 19:30:00 2022-06-23 01:30:00
SA1      2022-06-15 05:30:00 2022-06-23 00:00:00
VIC1     2022-06-19 18:00:00 2022-06-22 22:00:00

pd_rrp = 15100
                         min                 max
REGIONID                                        
NSW1     2022-06-01 18:00:00 2022-06-30 20:30:00
SA1      2022-06-01 17:30:00 2022-06-30 20:00:00
VIC1     2022-06-02 07:00:00 2022-06-30 20:00:00

 REGIONID     NSW1    SA1   VIC1
DAY                            
2022-06-08  431.0  419.0  383.0
2022-06-09  441.0  295.0  243.0
2022-06-10  607.0  291.0  275.0
2022-06-11  480.0  281.0  193.0
2022-06-12  463.0  482.0  317.0
2022-06-13  839.0  648.0  673.0
2022-06-14  296.0  156.0  229.0
2022-06-15  297.0  178.0  184.0
2022-06-16  264.0  251.0  227.0
2022-06-17  266.0  246.0  235.0
2022-06-18  262.0  190.0  196.0
2022-06-19  264.0  180.0  1

In [15]:
# Cap-price forecasts outside the window: genuine scarcity calls, or a broken forecast?
cap = p[p.pd_rrp == 15100]
print(cap.groupby(cap.DAY.dt.to_period("M")).size().to_string())

jun = cap[cap.DAY.between("2022-06-01", "2022-06-30")]
print(jun.groupby(["DAY", "REGIONID"]).size().unstack(fill_value=0).to_string())

DAY
2021-10     20
2021-11     35
2021-12     18
2022-01     44
2022-02      2
2022-03      1
2022-04      4
2022-05     94
2022-06    994
2025-06      5
Freq: M
REGIONID    NSW1  SA1  VIC1
DAY                        
2022-06-01     3   11     0
2022-06-02     7   10    15
2022-06-03    14    1     7
2022-06-07     4    5     0
2022-06-08     2    1     0
2022-06-13    11    0     4
2022-06-14    32    0     2
2022-06-15    43    9     9
2022-06-16    48   37    34
2022-06-17    48   48    44
2022-06-18    39   32    38
2022-06-19    26    4    12
2022-06-20    22   16    13
2022-06-21    33   40    28
2022-06-22    40   46    40
2022-06-24     8   13    11
2022-06-26     0    3     0
2022-06-27     2   14    15
2022-06-28     5    8     9
2022-06-29     0    7     3
2022-06-30     4    3    11


**Result: window = 13–24 Jun 2022** (1,728 half-hours, 0.66%).

- Every $300 half-hour falls inside it: first NSW 13 Jun 19:30, last NSW 23 Jun 01:30. 12 Jun and 25–28 Jun look like a normal (expensive) winter market.
- Inside the window, the forecast sits at the cap for 32–48 half-hours a day; outside it, at most 15 (evening peaks: genuine scarcity forecasts, kept).
- 24 Jun stays flagged: its forecast was made on 23 Jun, during the suspension.
- $15,100 was the cap only in FY2022, so the monthly counts don't detect cap forecasts in later years; the run check in section 7 does, and found none.

### C. Dec 2024 rooftop PV forecast outage

Target days 15–17 Dec 2024 used a PV forecast 5–53 h old. Did AEMO's predispatch degrade too?

In [16]:
dec = p[p.DAY.between("2024-12-08", "2024-12-22")]
err = dec.groupby(["DAY", "REGIONID"]).apply(lambda d: pd.Series({
    "pv_WAPE": wape(d.pv_actual_mw, d.pvf_powerpoe50),
    "dem_WAPE": wape(d.totaldemand, d.pd_totaldemand),
    "price_MAE": (d.rrp_mean - d.pd_rrp).abs().mean(),
})).unstack("REGIONID").round(2)
print(err.to_string())

           pv_WAPE             dem_WAPE             price_MAE                
REGIONID      NSW1   SA1  VIC1     NSW1   SA1  VIC1      NSW1     SA1    VIC1
DAY                                                                          
2024-12-08    0.12  0.07  0.05     0.04  0.10  0.04    536.69   30.34   21.45
2024-12-09    0.38  0.12  0.20     0.04  0.07  0.07    124.25   37.69   39.54
2024-12-10    0.03  0.06  0.09     0.01  0.04  0.03    281.25   23.35   24.78
2024-12-11    0.06  0.06  0.08     0.04  0.04  0.04     57.69   20.47   20.78
2024-12-12    0.05  0.05  0.11     0.02  0.04  0.04    762.45   40.64   44.95
2024-12-13    0.04  0.04  0.12     0.03  0.05  0.06   3771.57   47.91   52.59
2024-12-14    0.03  0.17  0.16     0.04  0.06  0.04     46.68   36.30   35.41
2024-12-15    0.08  0.10  0.04     0.03  0.05  0.04     43.79  122.49   33.30
2024-12-16    0.02  0.11  0.25     0.03  0.15  0.06     59.55   41.27   72.47
2024-12-17    0.03  0.03  0.21     0.05  0.07  0.04     76.79   

**Result: no flag.** PV, demand and price errors on 15–17 Dec sit within the range of the surrounding normal days (cloudy days produce similar PV errors). Prices were real market outcomes and only an input was stale, which is what a production model will face during any AEMO outage.

Price error in dollars is dominated by spikes (NSW 13 Dec: $3,772).